## Providing Summaries and Timestamps
Exploring how good GPT is at summarizing a full length professional tennis match from youtube, and providing timestamps for certain key moments in the match.

In [1]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Get the OpenAI API key
openai_api_key = os.getenv("OPENAI_API_KEY")

# Verify it loaded
if openai_api_key:
    print("OpenAI API key loaded successfully!")
else:
    print("OpenAI API key not found!")

OpenAI API key loaded successfully!


In [2]:
from langchain_openai import ChatOpenAI

llm_gpt_4 = ChatOpenAI(model="gpt-4.1-mini")

In [5]:
from langchain_community.document_loaders import YoutubeLoader

# 2025 Roland-Garros Final - Sinner vs Alcaraz
loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=ckbX699wngs", add_video_info=False
)

docs=loader.load()

transcript=docs[0].page_content

In [10]:
print(transcript)

[FetchedTranscriptSnippet(text='Heat.', start=15.36, duration=3.0), FetchedTranscriptSnippet(text='Heat.', start=18.4, duration=3.0), FetchedTranscriptSnippet(text='[music]', start=29.475, duration=2.02), FetchedTranscriptSnippet(text='Final miss.', start=36.719, duration=4.281), FetchedTranscriptSnippet(text='Espos.', start=51.039, duration=3.0), FetchedTranscriptSnippet(text='[music]', start=65.5, duration=2.02), FetchedTranscriptSnippet(text='[screaming]', start=78.6, duration=2.02), FetchedTranscriptSnippet(text='Good afternoon and welcome to Paris for', start=83.68, duration=6.24), FetchedTranscriptSnippet(text='one last time', start=87.439, duration=6.481), FetchedTranscriptSnippet(text='as the defending champion steps out here', start=89.92, duration=6.8), FetchedTranscriptSnippet(text='looking to go back to back on Philip', start=93.92, duration=5.641), FetchedTranscriptSnippet(text='Shatria.', start=96.72, duration=4.861), FetchedTranscriptSnippet(text='[music]', start=99.561,

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are a professional tennis analyst specializing in match analysis and commentary.

Your task: Analyze the provided tennis match transcript and create a structured summary.

TRANSCRIPT:
{video_transcript}

OUTPUT REQUIREMENTS:
1. Match Summary (25-75 words MAX. DO NOT go over 75 words for the match summary):
   - Players and final score
   - Key turning points
   - Notable performance highlights
   - Match significance/context

2. Significant Moments (with timestamps) - this is separate from the match summary, and provide NO MORE than 5 timestamps:
   - Match points
   - Break points at crucial moments
   - Exceptional rallies or shots
   - Momentum shifts
   - Set-defining games
   - Find the beginning 

FORMAT:
The match summary should be in paragraph form, to be insert into an HTML web page.
The timestamps should be in MM:SS or HH:MM:SS format and will be listed after the match summary.

TONE: Professional but engaging, suitable for tennis fans.
"""

prompt = PromptTemplate(
    input_variables=["video_transcript"],
    template=prompt_template,
)


In [12]:
chain = prompt | llm_gpt_4
result = chain.invoke({"video_transcript":transcript}).content

In [13]:
print(result)

**Match Summary:**  
In an epic Roland Garros final, Carlos Alcaraz defended his title against world number one Yanick Sinner with a thrilling 6-4, 1-6, 6-4, 6-7, 7-6 victory. The match showcased incredible shotmaking, resilience, and tactical brilliance, highlighted by multiple saved match points and shifting momentum. This historic clash, the longest men’s final in Roland Garros history, marked the first Grand Slam final between two players born in the 2000s and reinforced Alcaraz’s growing legacy.

**Significant Moments:**  
- 05:47:00 – Alcaraz saves three championship points to keep his title hopes alive.  
- 02:35:00 – Crucial break point earned by Alcaraz in the fourth set, swinging momentum.  
- 01:00:00 – First set-deciding break at 6-4, setting tone for the battle ahead.  
- 03:00:00 – Exceptional rally demonstrating high-quality baseline exchanges and stamina.  
- 05:14:00 – Match-winning tiebreak in the fifth set, cementing Alcaraz’s victory.


In [14]:
# 2025 Roland-Garros Final - Sinner vs Alcaraz
from youtube_transcript_api import YouTubeTranscriptApi

# Get timestamped transcript
video_id = "ckbX699wngs"

# Create an instance and fetch the transcript
ytt_api = YouTubeTranscriptApi()
transcript_data = ytt_api.fetch(video_id)

# transcript_data is a FetchedTranscript object - access the actual transcript list
transcript = transcript_data.snippets

# Now you have: [{'text': '...', 'start': 0.0, 'duration': 2.5}, ...]
print(f"Loaded {len(transcript)} transcript entries")
print(f"First entry: {transcript[0]}")

Loaded 4134 transcript entries
First entry: FetchedTranscriptSnippet(text='Heat.', start=15.36, duration=3.0)


In [15]:
chain = prompt | llm_gpt_4
result = chain.invoke({"video_transcript":transcript}).content

In [16]:
print(result)

The Roland Garros final featured Carlos Alcaraz defending his title against Yanick Sinner in an epic battle lasting over five hours. Alcaraz triumphed in five sets (6-4, 1-6, 6-4, 6-7, 7-6), showcasing incredible shot-making, mental toughness, and resilience by saving multiple match points. Both players pushed their limits with powerful groundstrokes and tactical variety on clay, marking one of the most memorable Grand Slam finals, notable for its intensity and historical significance as the first major final contested by players born in the 2000s.

Significant Moments:
- 04:02:18 – Match points saved by Alcaraz, turning the tide in the final set.
- 03:30:44 – Set-defining break in the third set, giving Alcaraz the lead.
- 02:21:18 – Momentum shift with Alcaraz rallying from being down multiple break points.
- 00:58:00 – Yanick Sinner breaks serve early, indicating competitive intensity.
- 05:14:52 – Deciding tie-break commencement, climaxing a historic contest.
